# Projet 9 - Bottleneck
## Préparation et contrôle des données

Les quatre fichiers CSV exportés depuis SQLite sont chargés. Leur contenu est
vérifié, les types sont corrigés et les clés sont contrôlées avant leur
utilisation dans un outil de visualisation.

La comparaison des méthodes d'extraction se trouve dans le rapport séparé.


## 1. Initialisation de l'environnement
### 1.1 Import des bibliothèques


In [1]:
from pathlib import Path

import pandas as pd

In [2]:
import sys
print(sys.version)

3.13.4 (tags/v3.13.4:8a526ec, Jun  3 2025, 17:46:04) [MSC v.1943 64 bit (AMD64)]


### 1.2 Chargement des données


In [3]:
DATA_DIR = Path(r"C:\Users\humbe\OneDrive\Desktop\Bi analyst\Projet-9\CSV")

df_finance_original = pd.read_csv(DATA_DIR / "Finance.csv", sep=";")
df_promotions_original = pd.read_csv(DATA_DIR / "Promo.csv", sep=";")
df_sales_original = pd.read_csv(DATA_DIR / "Sales.csv", sep=";")
df_web_original = pd.read_csv(DATA_DIR / "Web.csv", sep=";")

# Travail sur des copies afin de conserver les données importées.
df_finance = df_finance_original.copy()
df_promotions = df_promotions_original.copy()
df_sales = df_sales_original.copy()
df_web = df_web_original.copy()


## 2. Exploration initiale des données
### 2.1 Aperçu des tables


In [4]:
display(df_finance.head())
display(df_promotions.head())
display(df_sales.head())
display(df_web.head())

,Key_Finance,Date,SKU,Taxe_Statut,Onsale_Web,Price,Purchase_Price
0,2022-10_14692,2022-10-01,14692,taxable,1,"21,7","10,65"
1,2022-10_15328,2022-10-01,15328,taxable,1,55,"29,55"
2,2022-10_16515,2022-10-01,16515,taxable,1,"12,1","6,5"
3,2022-10_16585,2022-10-01,16585,taxable,1,"9,7","4,81"
4,2022-10_12869,2022-10-01,12869,taxable,1,"18,25","8,96"


,Key_Promo,Start_Date,End_Date,Sales_Offer,Price_Offer
0,2022-11_304,2022-11-21,2022-11-30,11,"33,99"
1,2022-11_531,2022-11-21,2022-11-30,8,"19,99"
2,2022-11_1360,2022-11-21,2022-11-30,7,"25,99"
3,2022-11_1364,2022-11-21,2022-11-30,4,"31,99"
4,2022-11_1366,2022-11-21,2022-11-30,6,"50,99"


,Key_Sales,Sales,Orders,Stock
0,2022-10_15325,20,162.0,142
1,2022-11_15325,39,NaN,103
2,2022-12_15325,45,NaN,58
3,2023-1_15325,11,48.0,95
4,2023-2_15325,10,NaN,85


,SKU,Virtuals,Downloadable,Rating_count,Average_rating,Post_author,Post_date,Post_date_gmt,Product_type,Segment,...,Post_status,Comment_status,Ping_status,Post_name,Post_modified,Post_modified_gmt,Post_parent,Guid,Post_type,Comment_count
0,38,0,0,8,"3,4",2,18-04-2022,18-04-2022,Vin,Vin,...,publish,closed,closed,emile-boeckel-cremant-brut-blanc-de-blancs,27-08-2022,27-08-2022,0,https://www.bottle-neck.fr/?post_type=product&...,product,8
1,41,0,0,1,"4,4",2,14-04-2022,14-04-2022,Vin,Vin,...,publish,closed,closed,marcel-windholtz-eau-de-marc-de-gewurztraminer,03-08-2022,03-08-2022,0,https://www.bottle-neck.fr/?post_type=product&...,product,1
2,304,0,0,0,NaN,2,13-02-2021,13-02-2021,Champagne,Spiritueux,...,publish,closed,closed,gosset-champagne-grande-reserve,27-08-2023,27-08-2023,0,https://www.bottle-neck.fr/?post_type=product&...,product,0
3,523,0,0,3,"3,7",2,06-04-2020,06-04-2020,Cognac,Spiritueux,...,publish,closed,closed,cognac-normandin-mercier-vfc,12-08-2022,12-08-2022,0,https://www.bottle-neck.fr/?post_type=product&...,product,3
4,531,0,0,5,"3,2",2,18-07-2021,18-07-2021,Champagne,Spiritueux,...,publish,closed,closed,champagne-petit-lebrun-fils-blanc-de-blancs-gr...,27-08-2023,27-08-2023,0,https://www.bottle-neck.fr/?post_type=product&...,product,5


### 2.2 Dimensions des jeux de données


In [5]:
display(df_finance.shape)
display(df_promotions.shape)
display(df_sales.shape)
display(df_web.shape)

(11323, 7)

(698, 5)

(9564, 4)

(761, 23)

### 2.3 Vérification des types de données


In [6]:
display(df_finance.dtypes)
display(df_promotions.dtypes)
display(df_sales.dtypes)
display(df_web.dtypes)

Key_Finance       object
Date              object
SKU                int64
Taxe_Statut       object
Onsale_Web         int64
Price             object
Purchase_Price    object
dtype: object

Key_Promo      object
Start_Date     object
End_Date       object
Sales_Offer     int64
Price_Offer    object
dtype: object

Key_Sales     object
Sales          int64
Orders       float64
Stock          int64
dtype: object

SKU                   int64
Virtuals              int64
Downloadable          int64
Rating_count          int64
Average_rating       object
Post_author           int64
Post_date            object
Post_date_gmt        object
Product_type         object
Segment              object
Post_title           object
Post_excerpt         object
Contenance           object
Post_status          object
Comment_status       object
Ping_status          object
Post_name            object
Post_modified        object
Post_modified_gmt    object
Post_parent           int64
Guid                 object
Post_type            object
Comment_count         int64
dtype: object

## 3. Vérification de la qualité des données
### 3.1 Recherche des valeurs manquantes

In [7]:
print("Valeurs manquantes - Finance")
display(df_finance.isna().sum())
print("Valeurs manquantes - Promotions")
display(df_promotions.isna().sum())
print("Valeurs manquantes - Sales")
display(df_sales.isna().sum())
print("Valeurs manquantes - Web")
display(df_web.isna().sum())


Valeurs manquantes - Finance


Key_Finance       0
Date              0
SKU               0
Taxe_Statut       0
Onsale_Web        0
Price             0
Purchase_Price    0
dtype: int64

Valeurs manquantes - Promotions


Key_Promo      0
Start_Date     0
End_Date       0
Sales_Offer    0
Price_Offer    0
dtype: int64

Valeurs manquantes - Sales


Key_Sales       0
Sales           0
Orders       5989
Stock           0
dtype: int64

Valeurs manquantes - Web


SKU                    0
Virtuals               0
Downloadable           0
Rating_count           0
Average_rating       183
Post_author            0
Post_date              0
Post_date_gmt          0
Product_type           0
Segment                0
Post_title             0
Post_excerpt           0
Contenance             0
Post_status            0
Comment_status         0
Ping_status            0
Post_name              0
Post_modified          0
Post_modified_gmt      0
Post_parent            0
Guid                   0
Post_type              0
Comment_count          0
dtype: int64

### 3.2 Recherche des doublons


In [8]:
print("Doublons Finance:", df_finance.duplicated().sum())
print("Doublons Promotions:", df_promotions.duplicated().sum())
print("Doublons Sales:", df_sales.duplicated().sum())
print("Doublons Web:", df_web.duplicated().sum())

Doublons Finance: 0
Doublons Promotions: 0
Doublons Sales: 0
Doublons Web: 0


## 4. Vérification des valeurs manquantes
### 4.1 Valeurs manquantes dans `Orders`

In [9]:
print("Nombre de lignes Sales :", len(df_sales))
print("Orders manquants :", df_sales["Orders"].isna().sum())
print("Pourcentage manquant :", round(df_sales["Orders"].isna().mean() * 100, 2), "%")
display(df_sales.loc[df_sales["Orders"].isna()].head(10))

Nombre de lignes Sales : 9564
Orders manquants : 5989
Pourcentage manquant : 62.62 %


,Key_Sales,Sales,Orders,Stock
1,2022-11_15325,39,NaN,103
2,2022-12_15325,45,NaN,58
4,2023-2_15325,10,NaN,85
5,2023-3_15325,11,NaN,74
7,2023-5_15325,19,NaN,98
8,2023-6_15325,24,NaN,74
10,2023-8_15325,22,NaN,117
11,2023-9_15325,34,NaN,83
14,2022-11_15328,4,NaN,19
15,2022-12_15328,5,NaN,14


### 4.2 Distinction entre les valeurs manquantes et la valeur `0`


In [10]:
print("Nombre de Orders = 0 :", (df_sales["Orders"] == 0).sum())
print("Valeurs différentes dans Orders :")
display(df_sales["Orders"].value_counts(dropna=False).sort_index())

Nombre de Orders = 0 : 549
Valeurs différentes dans Orders :


Orders
0.0       549
3.0        12
4.0         8
5.0         6
6.0        55
         ... 
180.0       1
188.0       1
202.0       1
242.0       1
NaN      5989
Name: count, Length: 116, dtype: int64

### 4.3 Répartition des valeurs manquantes de `Orders` par mois

Ce contrôle permet de vérifier si les valeurs manquantes sont présentes sur
tous les mois ou si elles concernent surtout certaines périodes.


In [11]:
# Extraction du mois présent dans Key_Sales.
mois = pd.to_datetime(
    df_sales["Key_Sales"].str.split("_").str[0],
    format="%Y-%m",
    errors="coerce"
)

controle_orders = (
    df_sales.assign(Mois=mois)
    .groupby("Mois", dropna=False)
    .agg(
        Nombre_lignes=("Orders", "size"),
        Orders_manquants=("Orders", lambda x: x.isna().sum()),
        Orders_zero=("Orders", lambda x: (x == 0).sum())
    )
    .sort_index()
)

controle_orders["Taux_manquants_%"] = (
    controle_orders["Orders_manquants"]
    / controle_orders["Nombre_lignes"]
    * 100
).round(2)

controle_orders.index = controle_orders.index.to_period("M")
display(controle_orders)


,Nombre_lignes,Orders_manquants,Orders_zero,Taux_manquants_%
Mois,,,,
2022-10,714,423,0,59.24
2022-11,714,237,55,33.19
2022-12,714,659,55,92.30
2023-01,714,423,27,59.24
2023-02,714,480,55,67.23
2023-03,714,416,55,58.26
2023-04,714,423,27,59.24
2023-05,761,481,55,63.21
2023-06,761,415,28,54.53


### 4.4 Conclusion sur les valeurs manquantes de `Orders`

Le contrôle met en évidence 5 989 valeurs manquantes dans `Orders`, soit
62,62 % des lignes. Comme la valeur 0 existe déjà dans 549 lignes, les valeurs
manquantes ne sont pas remplacées par 0.

Le taux varie selon les mois. Il passe de 33,19 % en novembre 2022 à plus de
92 % en décembre 2022 et en septembre 2023. Les valeurs manquantes ne sont donc
pas réparties de la même manière sur toute la période.

Ces valeurs sont conservées, car aucune autre source ne permet de retrouver
les commandes manquantes.


### 4.5 Analyse des valeurs manquantes de `Average_rating`

In [12]:
print("Nombre de lignes Web :", len(df_web))
print("Average_rating manquants :", df_web["Average_rating"].isna().sum())
print("Pourcentage manquant :", round(df_web["Average_rating"].isna().mean() * 100, 2), "%")
display(
    df_web.loc[
        df_web["Average_rating"].isna(),
        ["SKU", "Average_rating", "Post_title"]
    ].head(10)
)

Nombre de lignes Web : 761
Average_rating manquants : 183
Pourcentage manquant : 24.05 %


,SKU,Average_rating,Post_title
2,304,NaN,Champagne Gosset Grande Réserve
5,791,NaN,Marcel Windholtz Eau de Vie de Baie de Houx
6,793,NaN,Marcel Windholtz Eau de Vie de Coing
7,798,NaN,Marcel Windholtz Eau de Vie de Kirsch d'Alsace...
8,802,NaN,Marcel Windholtz Eau de Vie de Marc de Muscat
10,805,NaN,Marcel Windholtz Eau de Vie de Poire Williams ...
13,1360,NaN,Champagne Mailly Grand Cru Brut Réserve
14,1364,NaN,Champagne Mailly Grand Cru Brut Rosé
16,1662,NaN,Champagne Gosset Grand Rosé
19,2534,NaN,Marcel Windholtz Eau de Vie de Fraise


In [13]:
display(df_web.loc[df_web["Average_rating"].isna(), "Rating_count"].value_counts(dropna=False))

Rating_count
0    183
Name: count, dtype: int64

### 4.6 Conclusion sur les valeurs manquantes de `Average_rating`

Le contrôle identifie 183 valeurs manquantes, soit 24,05 % des lignes. Leur
`Rating_count` est toujours égal à 0 : aucune note moyenne n'est disponible
car ces produits n'ont reçu aucun avis.

Les valeurs manquantes sont conservées pour ne pas les confondre avec une note
moyenne égale à 0.


## 5. Conversion des types de données

Les types détectés par pandas sont comparés avec le dictionnaire de données,
puis les conversions nécessaires sont réalisées.


### 5.1 Conversion des types de la table `Finance`

Les identifiants sont convertis en texte, la date en datetime,
`Onsale_Web` en booléen et les prix en nombres décimaux.


In [14]:
df_finance["Key_Finance"] = df_finance["Key_Finance"].astype("string")
df_finance["SKU"] = df_finance["SKU"].astype("string")
df_finance["Taxe_Statut"] = df_finance["Taxe_Statut"].astype("string")
df_finance["Date"] = pd.to_datetime(df_finance["Date"], format="%Y-%m-%d", errors="coerce")

In [15]:
display(df_finance["Onsale_Web"].value_counts(dropna=False))

Onsale_Web
1    9893
0    1429
4       1
Name: count, dtype: int64

In [16]:
display(df_finance.loc[~df_finance["Onsale_Web"].isin([0, 1]), ["Key_Finance", "SKU", "Onsale_Web"]])

,Key_Finance,SKU,Onsale_Web
7814,2023-6_7087,7087,4


#### Vérification de `Onsale_Web`

`Onsale_Web` doit contenir uniquement 0 ou 1. Une valeur égale à 4 apparaît
pour le SKU 7087 ; ce produit est vérifié avant correction.


In [17]:
display(df_finance.loc[df_finance["SKU"] == "7087", ["Key_Finance", "Date", "SKU", "Onsale_Web"]])

,Key_Finance,Date,SKU,Onsale_Web
846,2022-10_7087,2022-10-01,7087,0
1717,2022-11_7087,2022-11-01,7087,0
2588,2022-12_7087,2022-12-01,7087,0
3459,2023-1_7087,2023-01-01,7087,0
4330,2023-2_7087,2023-02-01,7087,0
5201,2023-3_7087,2023-03-01,7087,0
6072,2023-4_7087,2023-04-01,7087,0
6943,2023-5_7087,2023-05-01,7087,0
7814,2023-6_7087,2023-06-01,7087,4
8685,2023-7_7087,2023-07-01,7087,0


In [18]:
display(df_web.loc[df_web["SKU"].astype("string") == "7087", ["SKU", "Post_title", "Post_status", "Post_type"]])

,SKU,Post_title,Post_status,Post_type


In [19]:
# Correction de la valeur à partir de la clé, pas du numéro de ligne.
# Le produit 7087 est absent de Web et vaut 0 pendant les douze autres mois.
masque_7087 = df_finance["Key_Finance"] == "2023-6_7087"
df_finance.loc[masque_7087, "Onsale_Web"] = 0
display(df_finance.loc[masque_7087])


,Key_Finance,Date,SKU,Taxe_Statut,Onsale_Web,Price,Purchase_Price
7814,2023-6_7087,2023-06-01,7087,taxable,0,30,"15,66"


In [20]:
df_finance["Onsale_Web"] = df_finance["Onsale_Web"].astype("boolean")

In [21]:
df_finance["Price"] = pd.to_numeric(
    df_finance["Price"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

df_finance["Purchase_Price"] = pd.to_numeric(
    df_finance["Purchase_Price"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

In [22]:
display(df_finance.dtypes)

Key_Finance       string[python]
Date              datetime64[ns]
SKU               string[python]
Taxe_Statut       string[python]
Onsale_Web               boolean
Price                    Float64
Purchase_Price           Float64
dtype: object

In [23]:
display(df_finance.isna().sum())

Key_Finance       0
Date              0
SKU               0
Taxe_Statut       0
Onsale_Web        0
Price             0
Purchase_Price    0
dtype: int64

### 5.2 Conversion des types de la table `Promotions`

In [24]:
df_promotions["Key_Promo"] = df_promotions["Key_Promo"].astype("string")
df_promotions["Start_Date"] = pd.to_datetime(
    df_promotions["Start_Date"], format="%Y-%m-%d", errors="coerce"
)
df_promotions["End_Date"] = pd.to_datetime(
    df_promotions["End_Date"], format="%Y-%m-%d", errors="coerce"
)
df_promotions["Price_Offer"] = pd.to_numeric(
    df_promotions["Price_Offer"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)
display(df_promotions.dtypes)
display(df_promotions.isna().sum())

Key_Promo      string[python]
Start_Date     datetime64[ns]
End_Date       datetime64[ns]
Sales_Offer             int64
Price_Offer           Float64
dtype: object

Key_Promo      0
Start_Date     0
End_Date       0
Sales_Offer    0
Price_Offer    0
dtype: int64

### 5.3 Conversion des types de la table `Sales`

In [25]:
df_sales["Key_Sales"] = df_sales["Key_Sales"].astype("string")
df_sales["Orders"] = df_sales["Orders"].astype("Int64")
display(df_sales.dtypes)
display(df_sales.isna().sum())

Key_Sales    string[python]
Sales                 int64
Orders                Int64
Stock                 int64
dtype: object

Key_Sales       0
Sales           0
Orders       5989
Stock           0
dtype: int64

### 5.4 Conversion des types de la table `Web`

In [26]:
# Mise au même type des SKU de Finance et Web pour permettre la relation.
df_web["SKU"] = df_web["SKU"].astype("string")
df_web["Virtuals"] = pd.to_numeric(
    df_web["Virtuals"], errors="coerce"
).astype("Int8")
df_web["Downloadable"] = pd.to_numeric(
    df_web["Downloadable"], errors="coerce"
).astype("Int8")

# Rating_count doit être un entier d'après le schéma.
df_web["Rating_count"] = pd.to_numeric(
    df_web["Rating_count"], errors="coerce"
).astype("Int64")

# Conservation de ces trois colonnes au format texte.
df_web["Post_author"] = df_web["Post_author"].astype("string")
df_web["Post_parent"] = df_web["Post_parent"].astype("string")
df_web["Comment_count"] = df_web["Comment_count"].astype("string")


In [27]:
df_web["Average_rating"] = pd.to_numeric(
    df_web["Average_rating"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

In [28]:
date_columns = [
    "Post_date",
    "Post_date_gmt",
    "Post_modified",
    "Post_modified_gmt"
]

for col in date_columns:
    df_web[col] = pd.to_datetime(
        df_web[col], dayfirst=True, errors="coerce")

In [29]:
text_columns = [
    "Product_type",
    "Segment",
    "Post_title",
    "Post_excerpt",
    "Contenance",
    "Post_status",
    "Comment_status",
    "Ping_status",
    "Post_name",
    "Guid",
    "Post_type"
]

for col in text_columns:
    df_web[col] = df_web[col].astype("string")

In [30]:
display(df_web.dtypes)
display(df_web.isna().sum())

SKU                  string[python]
Virtuals                       Int8
Downloadable                   Int8
Rating_count                  Int64
Average_rating              Float64
Post_author          string[python]
Post_date            datetime64[ns]
Post_date_gmt        datetime64[ns]
Product_type         string[python]
Segment              string[python]
Post_title           string[python]
Post_excerpt         string[python]
Contenance           string[python]
Post_status          string[python]
Comment_status       string[python]
Ping_status          string[python]
Post_name            string[python]
Post_modified        datetime64[ns]
Post_modified_gmt    datetime64[ns]
Post_parent          string[python]
Guid                 string[python]
Post_type            string[python]
Comment_count        string[python]
dtype: object

SKU                    0
Virtuals               0
Downloadable           0
Rating_count           0
Average_rating       183
Post_author            0
Post_date              0
Post_date_gmt          0
Product_type           0
Segment                0
Post_title             0
Post_excerpt           0
Contenance             0
Post_status            0
Comment_status         0
Ping_status            0
Post_name              0
Post_modified          1
Post_modified_gmt      1
Post_parent            0
Guid                   0
Post_type              0
Comment_count          0
dtype: int64

#### Vérification des dates de la table `Web`

Après la conversion, une valeur devient `NaT` dans `Post_modified` et
`Post_modified_gmt`. La copie d'origine permet de retrouver la date qui pose
problème.


In [31]:
display(df_web_original.loc[
        df_web["Post_modified"].isna() |
        df_web["Post_modified_gmt"].isna(),
        ["SKU", "Post_title", "Post_modified", "Post_modified_gmt"]
    ]
)

,SKU,Post_title,Post_modified,Post_modified_gmt
200,14596,David Duband Chambolle-Musigny 1er Cru Les Sen...,29/02/2022 15:25:02,29/02/2022 14:25:02


#### Conclusion sur les dates

La date 29/02/2022 apparaît pour le SKU 14596. Cette date n'existe pas, car
2022 n'est pas une année bissextile.

La valeur `NaT` est conservée, car aucune information ne permet de retrouver
la date correcte.


In [32]:
display(df_web.dtypes)
display(df_web.isna().sum())

SKU                  string[python]
Virtuals                       Int8
Downloadable                   Int8
Rating_count                  Int64
Average_rating              Float64
Post_author          string[python]
Post_date            datetime64[ns]
Post_date_gmt        datetime64[ns]
Product_type         string[python]
Segment              string[python]
Post_title           string[python]
Post_excerpt         string[python]
Contenance           string[python]
Post_status          string[python]
Comment_status       string[python]
Ping_status          string[python]
Post_name            string[python]
Post_modified        datetime64[ns]
Post_modified_gmt    datetime64[ns]
Post_parent          string[python]
Guid                 string[python]
Post_type            string[python]
Comment_count        string[python]
dtype: object

SKU                    0
Virtuals               0
Downloadable           0
Rating_count           0
Average_rating       183
Post_author            0
Post_date              0
Post_date_gmt          0
Product_type           0
Segment                0
Post_title             0
Post_excerpt           0
Contenance             0
Post_status            0
Comment_status         0
Ping_status            0
Post_name              0
Post_modified          1
Post_modified_gmt      1
Post_parent            0
Guid                   0
Post_type              0
Comment_count          0
dtype: int64

## 6. Vérification des clés et des relations

L'unicité des clés et les correspondances entre les lignes des différentes
tables sont vérifiées.


### 6.1 Vérification de l'unicité des clés

In [33]:
print("Doublons Key_Finance :",df_finance["Key_Finance"].duplicated().sum())

print("Doublons Key_Promo :",df_promotions["Key_Promo"].duplicated().sum())

print( "Doublons Key_Sales :",df_sales["Key_Sales"].duplicated().sum())

print("Doublons SKU Web :",df_web["SKU"].duplicated().sum())

Doublons Key_Finance : 0
Doublons Key_Promo : 0
Doublons Key_Sales : 0
Doublons SKU Web : 0


#### Conclusion sur l'unicité des clés

Aucun doublon n'est détecté dans `Key_Finance`, `Key_Promo`, `Key_Sales`
et `SKU`.


### 6.2 Correspondance entre les tables

In [34]:
df_sales["SKU"] = (
    df_sales["Key_Sales"].str.split("_").str[1].astype("string")
)
df_promotions["SKU"] = (
    df_promotions["Key_Promo"].str.split("_").str[1].astype("string")
)

In [35]:
print(
    "Clés Sales absentes de Finance :",
    (~df_sales["Key_Sales"].isin(df_finance["Key_Finance"])).sum()
)
print(
    "Clés Promotions absentes de Finance :",
    (~df_promotions["Key_Promo"].isin(df_finance["Key_Finance"])).sum()
)
print(
    "SKU Web absents de Finance :",
    (~df_web["SKU"].isin(df_finance["SKU"])).sum()
)
print(
    "Clés Finance sans ligne Sales :",
    (~df_finance["Key_Finance"].isin(df_sales["Key_Sales"])).sum()
)

Clés Sales absentes de Finance : 0
Clés Promotions absentes de Finance : 0
SKU Web absents de Finance : 0
Clés Finance sans ligne Sales : 1759


### 6.3 Lignes `Finance` sans correspondance dans `Sales`

Les produits hors ligne sont séparés des produits indiqués en vente sur le Web
mais absents de `Sales`.


In [36]:
finance_sans_sales = df_finance.loc[
    ~df_finance["Key_Finance"].isin(df_sales["Key_Sales"])
].copy()

finance_sans_sales["Mois"] = finance_sans_sales["Date"].dt.to_period("M")
finance_sans_sales_en_ligne = finance_sans_sales.loc[
    finance_sans_sales["Onsale_Web"]
].merge(
    df_web[["SKU", "Product_type", "Segment"]],
    on="SKU",
    how="left",
    validate="many_to_one"
)

print("Finance sans Sales :", len(finance_sans_sales))
print(
    "Dont produits hors ligne :",
    (~finance_sans_sales["Onsale_Web"]).sum()
)
print(
    "Dont produits indiqués en ligne :",
    finance_sans_sales["Onsale_Web"].sum()
)
print(
    "Nombre de SKU en ligne concernés :",
    finance_sans_sales_en_ligne["SKU"].nunique()
)

display(
    finance_sans_sales_en_ligne
    .groupby(["Product_type", "Segment"], dropna=False)
    .size()
    .rename("Nombre_lignes")
)
display(
    finance_sans_sales_en_ligne
    .groupby("Mois")
    .size()
    .rename("Nombre_lignes_manquantes")
)

Finance sans Sales : 1759
Dont produits hors ligne : 1430
Dont produits indiqués en ligne : 329
Nombre de SKU en ligne concernés : 47


Product_type  Segment    
Sans Alcool   Sans Alcool    329
Name: Nombre_lignes, dtype: int64

Mois
2022-10    47
2022-11    47
2022-12    47
2023-01    47
2023-02    47
2023-03    47
2023-04    47
Freq: M, Name: Nombre_lignes_manquantes, dtype: int64

#### Conclusion sur les correspondances `Finance`–`Sales`

Le contrôle met en évidence 1 430 lignes pour des produits hors ligne. Leur
absence dans `Sales` est donc cohérente.

Il met aussi en évidence 329 lignes concernant 47 produits `Sans Alcool`
indiqués en vente sur le Web, mais absents de `Sales` entre octobre 2022 et
avril 2023. Cela peut correspondre à un début de suivi en mai 2023, mais ces
absences ne peuvent pas être remplacées par des ventes égales à 0 sans
confirmation. Ces lignes sont donc conservées.


### 6.4 Vérification de la période

Les dates sont contrôlées afin de vérifier qu'elles sont comprises entre le
1er octobre 2022 et le 30 septembre 2023.


In [37]:
debut_periode = pd.Timestamp("2022-10-01")
fin_periode = pd.Timestamp("2023-09-30")

date_sales = pd.to_datetime(
    df_sales["Key_Sales"].str.split("_").str[0],
    format="%Y-%m",
    errors="coerce"
)

finance_hors_periode = df_finance.loc[
    ~df_finance["Date"].between(debut_periode, fin_periode)
]
sales_hors_periode = df_sales.loc[
    ~date_sales.between(debut_periode, fin_periode)
]
promotions_hors_periode = df_promotions.loc[
    (df_promotions["Start_Date"] < debut_periode)
    | (df_promotions["End_Date"] > fin_periode)
]

print("Période attendue :", debut_periode.date(), "au", fin_periode.date())
print("Lignes Finance hors période :", len(finance_hors_periode))
print("Lignes Sales hors période :", len(sales_hors_periode))
print("Lignes Promotions hors période :", len(promotions_hors_periode))

display(finance_hors_periode["Date"].value_counts().sort_index())
display(date_sales.loc[sales_hors_periode.index].value_counts().sort_index())

Période attendue : 2022-10-01 au 2023-09-30
Lignes Finance hors période : 871
Lignes Sales hors période : 761
Lignes Promotions hors période : 0


Date
2023-10-01    871
Name: count, dtype: int64

Key_Sales
2023-10-01    761
Name: count, dtype: int64

#### Conclusion sur la période

Le contrôle identifie 871 lignes dans `Finance` et 761 lignes dans `Sales`
pour octobre 2023. Aucune promotion ne dépasse la période prévue.

Octobre 2023 se trouve en dehors de la période demandée. Ces lignes sont
conservées dans `finance_hors_periode` et `sales_hors_periode`, mais ne sont
pas utilisées pour l'analyse de la période.


## 7. Vérifications métier

Recherche des prix ou quantités négatifs, des dates incohérentes et des valeurs
situées hors des limites attendues.

### 7.1 Vérification de la table `Finance`


In [38]:
print("Price <= 0 :", (df_finance["Price"] <= 0).sum())
print("Purchase_Price <= 0 :", (df_finance["Purchase_Price"] <= 0).sum())
print(
    "Prix de vente inférieur au prix d'achat :",
    (df_finance["Price"] < df_finance["Purchase_Price"]).sum()
)
print("Date minimum :", df_finance["Date"].min())
print("Date maximum :", df_finance["Date"].max())

Price <= 0 : 0
Purchase_Price <= 0 : 0
Prix de vente inférieur au prix d'achat : 0
Date minimum : 2022-10-01 00:00:00
Date maximum : 2023-10-01 00:00:00


#### Conclusion sur la table `Finance`

Aucun prix nul ou négatif n'est détecté. Aucun prix de vente n'est inférieur
au prix d'achat.


### 7.2 Vérification de la table `Promotions`

#### Vérification du dictionnaire

Le dictionnaire semble inverser les descriptions de `Sales_Offer` et
`Price_Offer`. D'après les types et les valeurs, `Sales_Offer` correspond
au nombre de produits vendus pendant la promotion et `Price_Offer` au prix
promotionnel. Cette interprétation est conservée pour les vérifications.


In [39]:
print(
    "Promotions avec une date de fin antérieure à la date de début :",
    (df_promotions["End_Date"] < df_promotions["Start_Date"]).sum()
)
print("Sales_Offer <= 0 :", (df_promotions["Sales_Offer"] <= 0).sum())
print("Price_Offer <= 0 :", (df_promotions["Price_Offer"] <= 0).sum())

Promotions avec une date de fin antérieure à la date de début : 0
Sales_Offer <= 0 : 0
Price_Offer <= 0 : 0


#### Comparaison avec le prix normal

Les promotions sont reliées aux données de `Finance` afin de comparer le prix
promotionnel avec le prix normal et le prix d'achat.


In [40]:
controle_promo = df_promotions.merge(
    df_finance[["Key_Finance", "Price", "Purchase_Price"]],
    left_on="Key_Promo",
    right_on="Key_Finance",
    how="left"
)

controle_promo = controle_promo.merge(
    df_sales[["Key_Sales", "Sales"]],
    left_on="Key_Promo",
    right_on="Key_Sales",
    how="left"
)

display(controle_promo.head())

,Key_Promo,Start_Date,End_Date,Sales_Offer,Price_Offer,SKU,Key_Finance,Price,Purchase_Price,Key_Sales,Sales
0,2022-11_304,2022-11-21,2022-11-30,11,33.99,304,2022-11_304,39.0,24.86,2022-11_304,24
1,2022-11_531,2022-11-21,2022-11-30,8,19.99,531,2022-11_531,22.5,13.78,2022-11_531,25
2,2022-11_1360,2022-11-21,2022-11-30,7,25.99,1360,2022-11_1360,29.5,17.55,2022-11_1360,19
3,2022-11_1364,2022-11-21,2022-11-30,4,31.99,1364,2022-11_1364,37.5,21.88,2022-11_1364,9
4,2022-11_1366,2022-11-21,2022-11-30,6,50.99,1366,2022-11_1366,59.0,35.45,2022-11_1366,19


In [41]:
print(
    "Promotions sans correspondance dans Finance :",
    controle_promo["Price"].isna().sum()
)
print(
    "Promotions sans correspondance dans Sales :",
    controle_promo["Sales"].isna().sum()
)

Promotions sans correspondance dans Finance : 0
Promotions sans correspondance dans Sales : 0


In [42]:
anomalies_promo = controle_promo.loc[
    controle_promo["Sales_Offer"] > controle_promo["Sales"]
]

print(
    "Prix promotionnels supérieurs au prix normal :",
    (controle_promo["Price_Offer"] > controle_promo["Price"]).sum()
)
print(
    "Prix promotionnels inférieurs au prix d'achat :",
    (controle_promo["Price_Offer"] < controle_promo["Purchase_Price"]).sum()
)
print(
    "Ventes promotionnelles supérieures aux ventes mensuelles :",
    len(anomalies_promo)
)
display(
    anomalies_promo[[
        "Key_Promo", "SKU", "Sales_Offer", "Sales", "Price_Offer", "Price"
    ]]
)

Prix promotionnels supérieurs au prix normal : 0
Prix promotionnels inférieurs au prix d'achat : 0
Ventes promotionnelles supérieures aux ventes mensuelles : 1


,Key_Promo,SKU,Sales_Offer,Sales,Price_Offer,Price
520,2023-9_13078,13078,7,1,29.59,37.0


#### Conclusion sur la table `Promotions`

Toutes les promotions correspondent à une ligne de `Finance` et de
`Sales`. Les dates et les prix sont cohérents.

Une anomalie apparaît pour la clé `2023-9_13078` : `Sales_Offer` vaut 7
alors que `Sales` vaut 1. Si `Sales_Offer` représente bien les ventes de la
promotion, cette valeur ne peut pas dépasser les ventes du mois. La valeur est
conservée et signalée pour vérification.


### 7.3 Vérification de la table `Sales`

In [43]:
print("Sales < 0 :", (df_sales["Sales"] < 0).sum())
print("Stock < 0 :", (df_sales["Stock"] < 0).sum())
print("Orders < 0 :", (df_sales["Orders"] < 0).sum())
print("Sales = 0 :", (df_sales["Sales"] == 0).sum())
print("Stock = 0 :", (df_sales["Stock"] == 0).sum())

Sales < 0 : 0
Stock < 0 : 0
Orders < 0 : 0
Sales = 0 : 11
Stock = 0 : 68


#### Conclusion sur la table `Sales`

Aucune valeur négative n'est détectée dans `Sales`, `Stock` et `Orders`.
Les valeurs égales à 0 sont conservées, car elles peuvent indiquer une absence
de vente ou une rupture de stock.


### 7.4 Vérification de la table `Web`

In [44]:
print("Rating_count < 0 :", (df_web["Rating_count"] < 0).sum())
print("Average_rating < 0 :", (df_web["Average_rating"] < 0).sum())
print("Average_rating > 5 :", (df_web["Average_rating"] > 5).sum())
print("Virtuals hors 0/1 :", (~df_web["Virtuals"].isin([0, 1])).sum())
print("Downloadable hors 0/1 :", (~df_web["Downloadable"].isin([0, 1])).sum())

Rating_count < 0 : 0
Average_rating < 0 : 0
Average_rating > 5 : 0
Virtuals hors 0/1 : 0
Downloadable hors 0/1 : 0


In [45]:
print("Valeurs de Post_status :")
display(df_web["Post_status"].value_counts(dropna=False))

print("Valeurs de Post_type :")
display(df_web["Post_type"].value_counts(dropna=False))

print("Valeurs de Product_type :")
display(df_web["Product_type"].value_counts(dropna=False))

print("Valeurs de Segment :")
display(df_web["Segment"].value_counts(dropna=False))

Valeurs de Post_status :


Post_status
publish    761
Name: count, dtype: Int64

Valeurs de Post_type :


Post_type
product    761
Name: count, dtype: Int64

Valeurs de Product_type :


Product_type
Vin                    658
Sans Alcool             47
Champagne               28
Whisky                  14
Cognac                   8
Huile d'olive            3
Gin                      2
Offres commerciales      1
Name: count, dtype: Int64

Valeurs de Segment :


Segment
Vin                    658
Spiritueux              52
Sans Alcool             47
Huile                    3
Offres commerciales      1
Name: count, dtype: Int64

In [46]:
display(pd.crosstab(df_web["Product_type"], df_web["Segment"]))

Segment,Huile,Offres commerciales,Sans Alcool,Spiritueux,Vin
Product_type,,,,,
Champagne,0,0,0,28,0
Cognac,0,0,0,8,0
Gin,0,0,0,2,0
Huile d'olive,3,0,0,0,0
Offres commerciales,0,1,0,0,0
Sans Alcool,0,0,47,0,0
Vin,0,0,0,0,658
Whisky,0,0,0,14,0


#### Conclusion sur la table `Web`

Aucune valeur numérique hors des limites attendues n'est détectée. Les notes
sont comprises entre 0 et 5. `Virtuals` et `Downloadable` contiennent
uniquement 0 ou 1. La correspondance entre `Product_type` et `Segment` est
également vérifiée.


### 7.5 Vérification des champs texte vides

Recherche des champs qui contiennent une chaîne vide ou uniquement des
espaces. Ces valeurs ne sont pas toujours comptées par `isna()`.


In [47]:
tables_controle = [
    ("Finance", df_finance),
    ("Promotions", df_promotions),
    ("Sales", df_sales),
    ("Web", df_web)
]

for nom_table, table in tables_controle:
    colonnes_texte = table.select_dtypes(
        include=["object", "string"]
    ).columns
    nombre_champs_vides = 0
    print(nom_table)

    for colonne in colonnes_texte:
        nombre = (
            table[colonne].astype("string").str.strip() == ""
        ).sum()
        if nombre > 0:
            print(colonne, ":", nombre)
            nombre_champs_vides += nombre

    if nombre_champs_vides == 0:
        print("Aucun champ texte vide")
    print()

Finance
Aucun champ texte vide

Promotions
Aucun champ texte vide

Sales
Aucun champ texte vide

Web
Aucun champ texte vide



### 7.6 Vérification du format des clés

Contrôle du format des clés et de leur cohérence avec les dates et les SKU.


In [48]:
mois_finance = (
    df_finance["Date"].dt.year.astype("string")
    + "-"
    + df_finance["Date"].dt.month.astype("string")
)
cle_finance_attendue = mois_finance + "_" + df_finance["SKU"]

mois_promo = (
    df_promotions["Start_Date"].dt.year.astype("string")
    + "-"
    + df_promotions["Start_Date"].dt.month.astype("string")
)
cle_promo_attendue = mois_promo + "_" + df_promotions["SKU"]

print(
    "Key_Finance différentes de Date + SKU :",
    (df_finance["Key_Finance"] != cle_finance_attendue).sum()
)
print(
    "Key_Promo différentes de Start_Date + SKU :",
    (df_promotions["Key_Promo"] != cle_promo_attendue).sum()
)

date_sales_controle = pd.to_datetime(
    df_sales["Key_Sales"].str.split("_").str[0],
    format="%Y-%m",
    errors="coerce"
)
print(
    "Dates incorrectes dans Key_Sales :",
    date_sales_controle.isna().sum()
)

Key_Finance différentes de Date + SKU : 0
Key_Promo différentes de Start_Date + SKU : 0
Dates incorrectes dans Key_Sales : 0


### 7.7 Contrôle final

Les principaux contrôles sont relancés après les conversions. La présence dans
la table `Web` des produits indiqués en vente sur le Web est également
vérifiée.


In [49]:
controle_final = pd.DataFrame({
    "Table": ["Finance", "Promotions", "Sales", "Web"],
    "Nombre de lignes": [
        len(df_finance),
        len(df_promotions),
        len(df_sales),
        len(df_web)
    ],
    "Valeurs manquantes": [
        df_finance.isna().sum().sum(),
        df_promotions.isna().sum().sum(),
        df_sales.isna().sum().sum(),
        df_web.isna().sum().sum()
    ],
    "Doublons complets": [
        df_finance.duplicated().sum(),
        df_promotions.duplicated().sum(),
        df_sales.duplicated().sum(),
        df_web.duplicated().sum()
    ]
})
display(controle_final)

sku_finance_en_ligne = (
    df_finance.loc[df_finance["Onsale_Web"], "SKU"]
    .drop_duplicates()
)
print(
    "SKU Finance en ligne absents de Web :",
    (~sku_finance_en_ligne.isin(df_web["SKU"])).sum()
)

,Table,Nombre de lignes,Valeurs manquantes,Doublons complets
0,Finance,11323,0,0
1,Promotions,698,0,0
2,Sales,9564,5989,0
3,Web,761,185,0


SKU Finance en ligne absents de Web : 0


## 8. Bilan de la préparation des données

Les quatre tables ont été chargées, converties et vérifiées. Les valeurs
manquantes sont conservées lorsqu'aucune autre information ne permet de les
compléter. Toutes les clés de `Sales` et de `Promo` correspondent à une ligne
de `Finance`.

Points retenus :

- 5 989 valeurs manquantes dans `Orders` ;
- 183 valeurs manquantes dans `Average_rating`, liées à l'absence d'avis ;
- une date impossible pour le SKU 14596, conservée sous la forme `NaT` ;
- une valeur égale à 4 dans `Onsale_Web`, corrigée à 0 après vérification ;
- 1 759 lignes de `Finance` sans ligne dans `Sales`, dont 1 430 produits hors
  ligne et 329 lignes concernant 47 produits `Sans Alcool` ;
- une promotion dont les ventes promotionnelles dépassent les ventes du mois ;
- 871 lignes de `Finance` et 761 lignes de `Sales` pour octobre 2023, en dehors
  de la période demandée.

Pour l'analyse, octobre 2023 est écarté. Les valeurs manquantes et l'anomalie
de promotion sont conservées afin de ne pas modifier les données sans
confirmation.


## 9. Préparation des données pour Power BI

Création des tables destinées à l'analyse sans modifier les DataFrames
contrôlés précédemment.

Les données temporelles sont limitées à la période demandée, du 1er octobre
2022 au 30 septembre 2023. Les lignes d'octobre 2023 restent conservées dans
les DataFrames de contrôle, mais ne sont pas intégrées aux fichiers utilisés
dans Power BI.


In [50]:
# Création de copies destinées à Power BI sans modifier les données contrôlées.
df_finance_powerbi = df_finance.loc[
    df_finance["Date"].between(debut_periode, fin_periode)
].copy()

df_sales_powerbi = df_sales.loc[
    date_sales.between(debut_periode, fin_periode)
].copy()

df_promotions_powerbi = df_promotions.copy()
df_web_powerbi = df_web.copy()


### 9.1 Création de la date mensuelle des ventes

La table `Sales` ne contient pas de colonne de date séparée. Le mois est
présent dans `Key_Sales`.

Une colonne `Date` est créée dans la copie destinée à Power BI afin de
faciliter les analyses temporelles. La clé d'origine reste inchangée.


In [51]:
# Extraction du mois présent dans Key_Sales pour créer une date exploitable.
df_sales_powerbi["Date"] = pd.to_datetime(
    df_sales_powerbi["Key_Sales"].str.split("_").str[0],
    format="%Y-%m",
    errors="coerce"
)

display(
    df_sales_powerbi[[
        "Key_Sales", "Date", "SKU", "Sales", "Orders", "Stock"
    ]].head()
)


,Key_Sales,Date,SKU,Sales,Orders,Stock
0,2022-10_15325,2022-10-01,15325,20,162,142
1,2022-11_15325,2022-11-01,15325,39,<NA>,103
2,2022-12_15325,2022-12-01,15325,45,<NA>,58
3,2023-1_15325,2023-01-01,15325,11,48,95
4,2023-2_15325,2023-02-01,15325,10,<NA>,85


### 9.2 Organisation des tables

Les quatre tables restent séparées pour le chargement dans Power BI.

Les données financières, les ventes, les promotions et les informations Web
sont déjà stockées dans des tables distinctes. Les champs sont atomiques et
les clés contrôlées sont uniques.

Aucune nouvelle normalisation n'est donc nécessaire et les relations prévues
dans le schéma fourni sont conservées.


## 10. Contrôle des données destinées à Power BI

Dernière vérification du périmètre temporel, de l'unicité des clés et des
relations entre les tables avant l'export.


In [52]:
controle_powerbi = pd.DataFrame({
    "Table": ["Finance", "Promotions", "Sales", "Web"],
    "Nombre de lignes": [
        len(df_finance_powerbi),
        len(df_promotions_powerbi),
        len(df_sales_powerbi),
        len(df_web_powerbi)
    ],
    "Doublons de clé": [
        df_finance_powerbi["Key_Finance"].duplicated().sum(),
        df_promotions_powerbi["Key_Promo"].duplicated().sum(),
        df_sales_powerbi["Key_Sales"].duplicated().sum(),
        df_web_powerbi["SKU"].duplicated().sum()
    ]
})

display(controle_powerbi)

,Table,Nombre de lignes,Doublons de clé
0,Finance,10452,0
1,Promotions,698,0
2,Sales,8803,0
3,Web,761,0


In [53]:
# Vérification des relations et du périmètre sur les données réellement exportées.
print(
    "Finance hors période :",
    (~df_finance_powerbi["Date"].between(
        debut_periode, fin_periode
    )).sum()
)

print(
    "Sales hors période :",
    (~df_sales_powerbi["Date"].between(
        debut_periode, fin_periode
    )).sum()
)

print(
    "Clés Sales absentes de Finance :",
    (~df_sales_powerbi["Key_Sales"].isin(
        df_finance_powerbi["Key_Finance"]
    )).sum()
)

print(
    "Clés Promotions absentes de Finance :",
    (~df_promotions_powerbi["Key_Promo"].isin(
        df_finance_powerbi["Key_Finance"]
    )).sum()
)


Finance hors période : 0
Sales hors période : 0
Clés Sales absentes de Finance : 0
Clés Promotions absentes de Finance : 0


## 11. Export des données préparées

Les quatre tables contrôlées sont exportées dans un dossier distinct des
fichiers d'origine. Ces fichiers constituent les sources utilisées dans
Power BI.


In [54]:
# Séparation des fichiers destinés à Power BI des exports d'origine.
EXPORT_DIR = DATA_DIR / "PowerBI"
EXPORT_DIR.mkdir(exist_ok=True)

df_finance_powerbi.to_csv(
    EXPORT_DIR / "Finance_PowerBI.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

df_promotions_powerbi.to_csv(
    EXPORT_DIR / "Promo_PowerBI.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

df_sales_powerbi.to_csv(
    EXPORT_DIR / "Sales_PowerBI.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

df_web_powerbi.to_csv(
    EXPORT_DIR / "Web_PowerBI.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("Exports Power BI créés dans :", EXPORT_DIR)


Exports Power BI créés dans : C:\Users\humbe\OneDrive\Desktop\Bi analyst\Projet-9\CSV\PowerBI


## 12. Conclusion

Les quatre tables ont été contrôlées et préparées pour leur utilisation dans
Power BI.

Les données d'octobre 2023 ont été conservées dans les données de contrôle mais
écartées du périmètre d'analyse. Les valeurs manquantes qui ne peuvent pas être
déduites ont été conservées et les anomalies nécessitant une validation métier
restent documentées.

Les fichiers exportés conservent la structure relationnelle fournie. Ils
peuvent maintenant être chargés dans Power BI pour créer le modèle de données,
les KPI et les analyses nécessaires au tableau de bord.